In [ ]:
import pandas as pd
import requests
import json
import numpy as np
from sqlalchemy import create_engine,text
import os
from dotenv import load_dotenv

def extract_data():
    url = "https://remotive.com/api/remote-jobs"
    try:
        response = requests.get(url)
        response.raise_for_status
        json_file = response.json()

        with open(r"raw/remotive_data.json",'w') as file:
            json.dump(json_file,file,indent=4)
            return json_file
    except Exception as e:
        print(f'Erro ao buscar os dados:{e}')

data = extract_data()



In [ ]:
# print(type(data))
# print(data.keys())
# print(data.values()

# # Exploração e definição do dataframe:
dados = data["jobs"]

# TRANSFORMAÇÃO DOS DADOS:

def trat_dados_origem(dados):
    """Realiza o tratamento geral da base de dados. como mundança de tipos e padronização de valores."""
    df = pd.DataFrame(dados)
    df_origem = df.copy()
    df_origem.columns

    columns_drop = ['company_logo_url','company_logo','url','description']
    df_origem = df_origem.drop(columns=columns_drop)

     # definindo moeda
    df_origem["currency"] = df_origem["salary"].apply(lambda x: "USD" if isinstance(x, str) and "$" in x else np.nan)

    # tratando inconsistencias no salario 
    df_origem["salary"] = (
        df_origem["salary"]
        .str.replace("OTE", "", regex=False)
        .str.strip()
    )
    
    # separando periodo
    hour = df_origem["salary"].str.contains(r'\b(hour|hr)\b',case=False,na=False)
    df_origem["period"] = np.where(hour,"hour","year")
    
    # separando faixa salarial:
    faixa = df_origem["salary"].str.split(r'\s*-\s*', n=1, expand=True)
    parte_min = faixa[0]
    parte_max = faixa[1]

    def extrai_valor(serie):
        """Faz a extração dos números na string, para determinar os valores minimo e maximo da faixa salarial."""
        # pega número (com , ou . decimal) + 'k' opcional
        num = serie.str.extract(r'(\d+(?:[.,]\d+)?)\s*([kK])?')
        valor = num[0].str.replace(',', '.', regex=False).astype(float)
        valor = np.where(num[1].notna(), valor * 1000, valor)
        return valor

    df_origem["salary_min"] = extrai_valor(parte_min)
    df_origem["salary_max"] = extrai_valor(parte_max)
    
    df_origem["salary_max"] = df_origem["salary_max"].fillna(df_origem["salary_min"])


    # tratando coluna de data:
    df_origem["publication_date"] = pd.to_datetime(df_origem["publication_date"],errors='coerce')


    return df_origem


# Explosão da lista de habilidades da vaga, pra posterior manipulação:

df_exploded_skills = trat_dados_origem(dados).explode("tags",ignore_index=True)



# Modelagem dos dados:

# Tratamento

def criar_dim_empresa(df_origem):
    dim_empresa = (
        df_origem[["company_name"]]
        .drop_duplicates()
        .reset_index(drop=True))
    dim_empresa = dim_empresa.rename(columns={"company_name":"nome_empresa"})
    return dim_empresa


def criar_dim_skill(df_exploded):

    dim_skill = (
    df_exploded[["tags"]]
    .drop_duplicates()
    .reset_index(drop=True)

)
    dim_skill = dim_skill.rename(columns={"tags":"nome_skill"})
    dim_skill["nome_skill"] = dim_skill["nome_skill"].fillna("Não especificado")
    return dim_skill

def criar_dim_categoria(df_origem):
    dim_categoria = (
        df_origem[["category"]]
        .drop_duplicates()
        .reset_index(drop=True)

    )
    dim_categoria = dim_categoria.rename(columns={"category":"nome_categoria"})
    return dim_categoria 

def criar_dim_localizacao(df_origem):
    dim_local = ( 
        df_origem[["candidate_required_location"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    dim_local = dim_local.rename(columns={"candidate_required_location":"localizacao_candidato"})
    return dim_local


df_origem = trat_dados_origem(dados)

dim_empresa = criar_dim_empresa(df_origem)
dim_skill = criar_dim_skill(df_exploded_skills)
dim_local = criar_dim_localizacao(df_origem)
dim_categoria = criar_dim_categoria(df_origem)



In [ ]:
# CARGA DOS DADOS:
load_dotenv(override=True)

tabelas_dim = {"dim_categoria":dim_categoria,
               "dim_empresa":dim_empresa,"dim_skill":dim_skill,
               "dim_local":dim_local}

def conectar_banco():
    """Realiza a conexão com o banco de dados."""
    host = os.getenv("host")
    database = os.getenv("database")
    user = os.getenv("usuario")
    port = os.getenv("port")
    password = os.getenv("senha_banco")
    url = f"postgresql://{user}:{password}@{host}:{port}/{database}"

    engine = create_engine(url)

    try:
        with engine.connect():
            return engine
    except Exception as e:
        print(f"A conexão falhou:{e}")
        raise


    
def carregar_dimensao(df,nome_tabela,conn):
        """Define o script SQL responsável por iterar as linhas de cada dataframe e inserir no banco de dados apenas registros novos."""

        coluna = df.columns[0]
        sql = f"""
        INSERT INTO {nome_tabela} ({coluna})
        VALUES (:valor)
        ON CONFLICT ({coluna}) DO NOTHING
        """
        for _, row in df.iterrows():
            conn.execute(
            text(sql),
            {"valor": row[coluna]}
        )


def carregar_dimensoes(tabelas_dim):
    """Carrega todas tabelas no banco de dados."""

    engine = conectar_banco()

    with engine.begin() as conn:
        for nome_tabela, df in tabelas_dim.items():
            carregar_dimensao(df, nome_tabela, conn)
        


In [ ]:
tabelas_dim = {"dim_categoria":dim_categoria,
               "dim_empresa":dim_empresa,"dim_skill":dim_skill,
               "dim_local":dim_local}

engine = conectar_banco()


dim_empresa =  pd.read_sql("SELECT * FROM dim_empresa", engine)
dim_categoria= pd.read_sql("SELECT * FROM dim_categoria", engine)
dim_skill = pd.read_sql("SELECT * FROM dim_skill", engine)
dim_local =  pd.read_sql("SELECT * FROM dim_local", engine)


def criar_fato():
    
    fato_vagas = df_origem.merge(
        dim_empresa,
        how='left',
        left_on='company_name',
        right_on='nome_empresa'
    )

    fato_vagas = fato_vagas.merge(
        dim_categoria,
        how='left',
        left_on='category',
        right_on='nome_categoria'
    )
    fato_vagas = fato_vagas.merge(
        dim_local,
        how='left',
        left_on='candidate_required_location',
        right_on='localizacao_candidato'
    )
    columns = ["id","title","job_type","publication_date","salary_min","currency","salary_max","id_empresa","id_categoria","id_local","period"]
    
    fato_vagas = fato_vagas[columns]
    fato_vagas = fato_vagas.rename(columns={"id": "vaga_id","id_empresa":"empresa_id","id_categoria":"categoria_id","id_local":"localizacao_id"})
    return fato_vagas

def carregar_fato_vagas(tabela_fato,nome_tabela):
    engine = conectar_banco()
    colunas = tabela_fato.columns.to_list()
    coluna_constraint = "vaga_id"
    colunas_sql = ", ".join(colunas)
    valores_sql = ", ".join([f":{col}" for col in colunas])

    with engine.begin() as conn:
        sql = f"""
        INSERT INTO {nome_tabela} ({colunas_sql})
        VALUES ({valores_sql})
        ON CONFLICT ({coluna_constraint}) DO NOTHING
        """
        for _, row in tabela_fato.iterrows():
            conn.execute(
            text(sql),
            row.to_dict()
        )
            

            
fato = criar_fato()
carregar_fato_vagas(fato,'fato_vaga')
fato.columns.to_list()


['vaga_id',
 'title',
 'job_type',
 'publication_date',
 'salary_min',
 'currency',
 'salary_max',
 'empresa_id',
 'categoria_id',
 'localizacao_id',
 'period']